Attention Is All You Need

Published: Jun 2017

BERT: Bidirectional Encoder Representations from Transformers

Published: Oct 2018

![architecture](static/x-behrt-review/bert.png)

Full Transformer Models:

    Goal: Transform an input sequence into a completely different output sequence (sequence-to-sequence tasks like translation or summarization)

Decoder-Only Models:

    Goal: Predict a new output sequence in response to an input sequence

Encoder-Only Models:

    Goal: Make predictions about words within an input sequence

BEHRT: Bidirectional Encoder Representations from Transformers for EHR

Published: July 2019


![architecture](static/x-behrt-review/behrt-fig1.png)


![architecture](static/x-behrt-review/behrt-fig2.png)



Note:

- Token embeddings are a simple `SUM(embedding(diagnosis_code), embedding(age), embedding(position))` for each diagnosis token in the sequence.
- Visits are separated by `[SEP]` token
- No codes for medications, labs, or procedures are used in the original BEHRT model.

Multimodal BEHRT: Multimodal Bidirectional Encoder Representations from Transformers for EHR

Published: Sep 2024

![architecture](static/x-behrt-review/m-behrt.jpg)

Pros:
- Tabular token embeddings have evolved since the original BEHRT paper, and now cover:

    - diagnoses, medications, labs or procedures codes
    - values for continuous features (e.g., lab results)
    - time difference between visits
    - measurement difference between visits

Cons:
- Visits are still separated by `[SEP]` token
- Position and Segment embeddings are still used, which may not be ideal for irregularly spaced visits
- Likely degenerate final cross-attention module, but it likely could be fixed by a better design


DT-BEHRT: Disease Trajectory-aware Transformer for Interpretable Patient Representation Learning

Published: Mar 2026


![architecture](static/x-behrt-review/dt-behrt-architecture-final.jpg)

With legend:
```
SR = fine-grained event-level context
DA = chapter/system-level disease abstraction
DP = visit-level temporal progression
PR = final attention-based fusion of these summaries
```

and assuming:

```text
L   = 3 high-level DT-BEHRT layers
L_G = 2 GAT blocks inside each DP update
```

with notation:

```text
S = SR tokens = [SEQ] + diagnosis/med/lab/procedure tokens
A = DA tokens = ICD chapter/group tokens
P = DP tokens = visit nodes

D = hidden_dim
N = number of ordinary SR code tokens
G = number of DA/group tokens
V = number of visits / DP tokens
```

The computation flow is as follows:

Initial tensors:

```text
S^(0): [1 + N, D]
A^(0): [G, D]
P^(0): [V, D]
```

The Transformer-side tensor is conceptually:

```text
X^(k) = [S^(k) || A^(k) || P^(k)]
shape = [1 + N + G + V, D]
```

But the tokens are **not equally connected**. The mask determines which rows can read which columns.

## Mask structure

Rows are **queries**. Columns are **keys/values**.

A compact block view:

```text
query \ key      S tokens        A tokens        P tokens
------------------------------------------------------------
S tokens         allowed         blocked         blocked

A tokens         chapter-only    self-only       blocked

P tokens         blocked         blocked         self-only / GNN-updated
```

Meaning:

```text
S → S:
  ordinary SR self-attention over [SEQ] and code tokens.

S → A, S → P:
  blocked. SR tokens do not read DA or DP tokens inside the Transformer.

A → S:
  allowed only for diagnosis codes that belong to that DA token's ICD chapter.

A → A:
  only self-attention is allowed; DA tokens do not freely mix with each other.

A → P:
  blocked.

P → anything:
  in the open-source implementation, visit/DP tokens are effectively not updated by Transformer attention;
  they are updated by the GNN path instead.
```

This matches the paper’s description that DA attention is chapter-restricted, and the implementation’s attention mask uses `True = blocked`, `False = allowed`; code/CLS token rows block group and visit tokens, while group-token rows are blocked by default and selectively unblocked only toward corresponding diagnosis codes. ([arXiv][1])

## Layer 0: input construction

```text
S^(0):
  code embedding + type embedding + visit-index embedding

A^(0):
  learned ICD chapter/group token embedding
  appended once per qualifying chapter

P^(0):
  one visit token per visit
  initialized from age/visit embedding
```

The paper says DA tokens are appended to the flattened visit-major vector when a chapter threshold is satisfied, and DP graph visit nodes are initialized with age embeddings while diagnosis graph nodes use corresponding SR embeddings. ([arXiv][1])

## High-level layer 1

```text
1. SR/DA Transformer update:
   [S^(0), A^(0), P^(0)] + mask
   → S^(1), A^(1), P^(0)
```

But due to the mask:

```text
S^(1):
  updated from S^(0) only

A^(1):
  updated from diagnosis codes in the same ICD chapter
  plus itself

P^(0):
  essentially unchanged by Transformer attention
```

Then DP update:

The DP graph is built using the **current diagnosis embeddings** from `S^(1)`:

```text
diagnosis occurrence nodes = diagnosis-code embeddings selected from S^(1)
virtual visit nodes        = P^(0)
```

Edges connect:

```text
visit_t ↔ diagnosis occurrences in visit_t
visit_t → visit_{t+1}
```

Then the DP module applies `L_G = 2` GAT message-passing blocks:

```text
P^(0) → P^(1)
```

So after layer 1:

```text
S^(1): refined event-level sequence tokens
A^(1): refined chapter-level summaries
P^(1): refined visit/progression tokens
```

## High-level layer 2

```text
1. Transformer input:
   [S^(1) || A^(1) || P^(1)]

2. Same mask:
   S reads S
   A reads chapter-matched diagnosis codes
   P remains GNN-owned

3. Output:
    S^(2): further contextualized SR token embeddings
    A^(2): further contextualized DA token embeddings
    P^(1): current visit / DP token embeddings

The DP graph is rebuilt or refreshed using the current hidden states:

diagnosis occurrence nodes = diagnosis-code embeddings selected from S^(2)
virtual visit nodes        = P^(1)
```

The graph topology is still determined by the patient trajectory:

```text
visit_t ↔ diagnosis occurrences in visit_t
visit_t → visit_{t+1}
```

Then `L_G = 2` GAT blocks update the visit nodes:

```text
P^(1) → P^(2)
```

So after layer 2:

```text
S^(2), A^(2), P^(2)
```


## High-level layer 3

```text
1. Transformer input:
   [S^(2) || A^(2) || P^(2)]

2. Same masked Transformer update:
   S^(3), A^(3)

3. DP update:
    The DP graph uses the current diagnosis embeddings from `S^(3)`:

    diagnosis occurrence nodes = diagnosis-code embeddings selected from S^(3)
    virtual visit nodes        = P^(2)
```

The same patient-specific graph structure is used:

```text
visit_t ↔ diagnosis occurrences in visit_t
visit_t → visit_{t+1}
```

Then `L_G = 2` GAT message-passing blocks update the visit nodes:

```text
P^(2) → P^(3)
```

Final representations:

```text
S^(3): final SR tokens
A^(3): final DA tokens
P^(3): final DP visit tokens
```

## Patient Representation

At the end:

```text
PR = attention-pooling query from [SEQ]
     over selected summary tokens:
     [SEQ], DA tokens, DP tokens
```

In the implementation, this is done by `CLSQueryMHA`: `[CLS]/[SEQ]` is the query, and allowed key/value token types are `[CLS]`, `group`, and `visit`. Ordinary code tokens are not directly used in the final pooling set. ([GitHub][2])

So the brief summary is:

```text
SR tokens:
  updated by Transformer over SR tokens.

DA tokens:
  updated by Transformer, but only by attending to diagnosis codes
  in the same ICD chapter.

DP tokens:
  updated mainly by GNN/GAT over the visit-diagnosis graph.

PR:
  final attention-based fusion of [SEQ], DA, and DP summaries.
```

The key point: **SR, DA, and DP share the same hidden dimension and may be concatenated into one tensor, but the mask and GNN path keep their computational roles separate.**

[1]: https://arxiv.org/html/2603.10180v1 "DT-BEHRT: Disease Trajectory-aware Transformer for Interpretable Patient Representation Learning"
[2]: https://raw.githubusercontent.com/GatorAIM/DT-BEHRT/main/src/heterogt/model/layer.py "raw.githubusercontent.com"


### Patient Representation (PR) and Classification Heads

The **PR layer** produces the final patient-level representation by concatenating two vectors:

```text
PR = concat(
    h_[SEQ],
    AttnPool(query = h_[SEQ],
             keys/values = [h_[SEQ], H_DA, H_DP])
)
````

where:

```text
h_[SEQ]     : [B, D]
AttnPool(.) : [B, D]

PR          : [B, 2D]
```

So the final representation is split into two equal parts:

```text
50% = final [SEQ] embedding
50% = attention-pooled summary over [SEQ], DA, and DP tokens
```

The **classification head** receives this `[B, 2D]` tensor.

For binary prediction tasks:

```text
[B, 2D] → Linear(2D, 2D) → ReLU → Linear(2D, 1)
```

Output:

```text
logits: [B, 1]
```

A sigmoid can be applied afterward to obtain probabilities:

```text
probability = sigmoid(logit)
```

For multi-label or multi-class prediction tasks with `C` target labels:

```text
[B, 2D] → Linear(2D, 2D) → ReLU → Linear(2D, C)
```

Output:

```text
logits: [B, C]
```

The classification head produces **logits**, while the task-specific loss function determines how those logits are interpreted.

```

